In [1]:
import httpx
import pprint as pp
import base64
import json
from pprint import pprint
from refund_lab.client import decode_cursor
from refund_lab.utils import b64decode_relaxed

PORT = "8088"
API_URL = "http://127.0.0.1:" + PORT
DEV_TOKEN = "rl_live_8f2c1d94e6b74a03"


def auth_to_API(client: httpx.Client):
    resp = client.post(API_URL + "/v1/auth/token",
                   headers={"Authorization": f"Bearer {DEV_TOKEN}"})
    live_token = resp.json()["access_token"]
    client.headers["Authorization"] = f"Bearer {live_token}"

def byte_decode(cursor) -> dict | None:
    return json.loads(b64decode_relaxed(cursor))

In [2]:
client = httpx.Client(base_url=API_URL, timeout=10,
                      mounts={"all://localhost": None,
                              "all://127.0.0.1": None})
auth_to_API(client)

as_of = "2027-01-31T00:00:00"
since = "2026-01-15T00:00:00"
until = "2026-01-25T00:00:00"

response = client.get(API_URL + "/v1/customers",
                     params={"limit":50,
                             "as_of":as_of,
                             "since":since,
                             "until":until})
cursor = response.json()["cursor"]
next_cursor = response.json()["next_cursor"]

In [3]:
pprint(decode_cursor(cursor))
pprint(decode_cursor(next_cursor))

None
{'as_of': '2026-07-14T08:40:27',
 'entity': 'customers',
 'last_seen_value': '2026-01-18T16:16:18',
 'prev_last_seen_value': None,
 'prev_rows_served': None,
 'rows_served': 8676,
 'since': '2026-01-15T00:00:00',
 'total_count': 125,
 'until': '2026-01-25T00:00:00'}


In [11]:
response.json()["data"][0]

{'customer_id': '0008646',
 'valid_from': '2026-01-15T06:29:14',
 'signup_date': '2026-01-15',
 'country': 'DE',
 'city': 'Düsseldorf',
 'segment': 'enterprise',
 'lifetime_value_cents': 3730,
 'is_active': 1,
 'version': 1,
 'knowledge_time': '2026-01-15T07:40:14',
 'valid_to': None}